# 01 U-Net Baseline for Urban Change Detection

This notebook implements a complete U-Net baseline for the SYSU-CD style dataset structure.

## Goals
- Load bi-temporal remote sensing images from the predefined `train`, `val`, and `test` folders
- Concatenate `time1` and `time2` into a 6-channel input
- Train a U-Net model for binary change detection
- Evaluate the model on the test set using IoU, F1-score, Precision, and Recall
- Export standardized outputs for later group comparison

## Standardized Outputs
This notebook will save the following files under `outputs/unet/`:
- `metrics.csv`
- `training_history.csv`
- `config_used.yaml`
- `shared_test_ids.txt`
- `test_predictions/`
- `sample_visuals/`

In [ ]:
# Standard library imports
import os
import sys
import time
import random
from pathlib import Path

# Third-party imports
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Add the project root to Python path
# This allows the notebook to import files from the src/ folder
project_root = Path.cwd().resolve().parent
sys.path.append(str(project_root))

# Import the dataset utility you already created
from src.data_utils import create_dataloader

# Define the config path
config_path = project_root / "configs" / "unet.yaml"

# Load YAML config
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Resolve key paths relative to the project root
data_root = project_root / config["data_root"]
output_dir = project_root / config["output_dir"]
model_save_path = output_dir / config["model_save_name"]

print("Project root:", project_root)
print("Data root:", data_root)
print("Output dir:", output_dir)
print("Model save path:", model_save_path)
print("Loaded config:", config)

In [ ]:
# Set a fixed random seed for reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(config["seed"])

# Select device automatically
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Create all required output folders
output_dir.mkdir(parents=True, exist_ok=True)

test_predictions_dir = output_dir / "test_predictions"
sample_visuals_dir = output_dir / "sample_visuals"

test_predictions_dir.mkdir(parents=True, exist_ok=True)
sample_visuals_dir.mkdir(parents=True, exist_ok=True)

print("Created output folders successfully.")

## Load Dataset Splits

We directly use the predefined `train`, `val`, and `test` folders from the dataset.
This ensures that all models are trained and evaluated on the same split.

In [ ]:
# Create dataloaders using the folder-based split structure
train_dataset, train_loader = create_dataloader(
    root_dir=str(data_root),
    split=config["train_split"],
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=0
)

val_dataset, val_loader = create_dataloader(
    root_dir=str(data_root),
    split=config["val_split"],
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0
)

test_dataset, test_loader = create_dataloader(
    root_dir=str(data_root),
    split=config["test_split"],
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
# Inspect one sample from the training set
sample = train_dataset[0]

print("Sample ID:", sample["id"])
print("Image shape:", sample["image"].shape)
print("Mask shape:", sample["mask"].shape)
print("Image dtype:", sample["image"].dtype)
print("Mask dtype:", sample["mask"].dtype)
print("Image min/max:", sample["image"].min().item(), sample["image"].max().item())
print("Mask unique values:", torch.unique(sample["mask"]))

## Visualize a Sample

The first 3 channels correspond to `time1`, and the last 3 channels correspond to `time2`.
The mask is a binary change map.

In [ ]:
# Visualize one training sample
sample = train_dataset[0]
image = sample["image"].numpy()
mask = sample["mask"].squeeze(0).numpy()

time1_img = image[:3].transpose(1, 2, 0)
time2_img = image[3:].transpose(1, 2, 0)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(time1_img)
axes[0].set_title("Time 1")
axes[0].axis("off")

axes[1].imshow(time2_img)
axes[1].set_title("Time 2")
axes[1].axis("off")

axes[2].imshow(mask, cmap="gray")
axes[2].set_title("Label")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Define the U-Net Baseline

We use a standard U-Net architecture with:
- 6 input channels
- 1 output channel
- encoder-decoder structure with skip connections

The output is a pixel-wise binary change map.

In [ ]:
# Define a basic double convolution block
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# Define a standard U-Net
class UNet(nn.Module):
    def __init__(self, in_channels=6, out_channels=1, base_channels=64):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = DoubleConv(base_channels * 4, base_channels * 8)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(base_channels * 8, base_channels * 16)

        self.up4 = nn.ConvTranspose2d(base_channels * 16, base_channels * 8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(base_channels * 16, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base_channels * 2, base_channels)

        self.out_conv = nn.Conv2d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        logits = self.out_conv(d1)
        return logits

In [ ]:
# Build the model and test one forward pass
model = UNet(
    in_channels=config["input_channels"],
    out_channels=config["num_classes"]
).to(device)

sample_batch = next(iter(train_loader))
images = sample_batch["image"].to(device)

with torch.no_grad():
    logits = model(images)

print("Input batch shape:", images.shape)
print("Output logits shape:", logits.shape)

## Define Loss and Evaluation Metrics

The main training loss is the combination of:
- Binary Cross-Entropy (BCE)
- Dice Loss

The evaluation metrics are:
- IoU
- F1-score
- Precision
- Recall

In [ ]:
# Define Dice loss
def dice_loss(logits, targets, eps=1e-8):
    probs = torch.sigmoid(logits)
    probs = probs.view(-1)
    targets = targets.view(-1)

    intersection = (probs * targets).sum()
    union = probs.sum() + targets.sum()

    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice


# Define the combined BCE + Dice loss
def bce_dice_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    dloss = dice_loss(logits, targets)
    return bce + dloss


# Convert logits into binary predictions
def binarize_predictions(logits, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    return preds


# Compute confusion matrix counts
def compute_confusion_counts(preds, targets):
    preds = preds.view(-1)
    targets = targets.view(-1)

    tp = ((preds == 1) & (targets == 1)).sum().item()
    fp = ((preds == 1) & (targets == 0)).sum().item()
    fn = ((preds == 0) & (targets == 1)).sum().item()
    tn = ((preds == 0) & (targets == 0)).sum().item()

    return tp, fp, fn, tn


# Compute final metrics from confusion counts
def compute_metrics_from_counts(tp, fp, fn, tn, eps=1e-8):
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    iou = tp / (tp + fp + fn + eps)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "iou": iou
    }

In [ ]:
# Train the model for one epoch
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0

    for batch in loader:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = bce_dice_loss(logits, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss


# Validate the model for one epoch
def validate_one_epoch(model, loader, device, threshold=0.5):
    model.eval()
    running_loss = 0.0

    total_tp, total_fp, total_fn, total_tn = 0, 0, 0, 0

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            masks = batch["mask"].to(device)

            logits = model(images)
            loss = bce_dice_loss(logits, masks)

            running_loss += loss.item() * images.size(0)

            preds = binarize_predictions(logits, threshold=threshold)
            tp, fp, fn, tn = compute_confusion_counts(preds, masks)

            total_tp += tp
            total_fp += fp
            total_fn += fn
            total_tn += tn

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics_from_counts(total_tp, total_fp, total_fn, total_tn)

    return epoch_loss, metrics

In [ ]:
# Save the exact config used for this run
with open(output_dir / "config_used.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

# Build model again to make sure the training starts from a clean state
model = UNet(
    in_channels=config["input_channels"],
    out_channels=config["num_classes"]
).to(device)

# Define optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"]
)

# Initialize training trackers
best_val_loss = float("inf")
best_epoch = -1
patience_counter = 0

history = []

## Training Loop

We train the U-Net model with early stopping based on validation loss.
The best model checkpoint is saved automatically.

In [ ]:
# Run the full training loop
for epoch in range(1, config["num_epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)

    val_loss, val_metrics = validate_one_epoch(
        model,
        val_loader,
        device,
        threshold=config["threshold"]
    )

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_iou": val_metrics["iou"]
    }
    history.append(record)

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val IoU: {val_metrics['iou']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

    # Save the best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), model_save_path)
        print(f"Saved best model at epoch {epoch}")
    else:
        patience_counter += 1

    # Early stopping
    if patience_counter >= config["patience"]:
        print(f"Early stopping triggered at epoch {epoch}")
        break

print("Training finished.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

In [ ]:
# Save training history as CSV
history_df = pd.DataFrame(history)
history_df.to_csv(output_dir / "training_history.csv", index=False)

print(history_df.head())

In [ ]:
# Plot training and validation loss curves
plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training History")
plt.legend()
plt.grid(True)
plt.show()

## Final Test Evaluation

After training, we load the best checkpoint and evaluate the model on the test set.
The final outputs include:
- IoU
- F1-score
- Precision
- Recall
- parameter count
- inference time

In [ ]:
# Load the best checkpoint before test evaluation
best_model = UNet(
    in_channels=config["input_channels"],
    out_channels=config["num_classes"]
).to(device)

best_model.load_state_dict(torch.load(model_save_path, map_location=device))
best_model.eval()

print("Loaded best model successfully.")

In [ ]:
# Evaluate on the test set
total_tp, total_fp, total_fn, total_tn = 0, 0, 0, 0
total_inference_time = 0.0
total_images = 0

with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)

        start_time = time.perf_counter()
        logits = best_model(images)
        end_time = time.perf_counter()

        total_inference_time += (end_time - start_time)
        total_images += images.size(0)

        preds = binarize_predictions(logits, threshold=config["threshold"])
        tp, fp, fn, tn = compute_confusion_counts(preds, masks)

        total_tp += tp
        total_fp += fp
        total_fn += fn
        total_tn += tn

test_metrics = compute_metrics_from_counts(total_tp, total_fp, total_fn, total_tn)

# Compute parameter count and average inference time
param_count = sum(p.numel() for p in best_model.parameters())
param_count_million = param_count / 1e6
avg_inference_time_ms = (total_inference_time / total_images) * 1000

print("Test metrics:", test_metrics)
print("Parameter count (M):", param_count_million)
print("Average inference time per image (ms):", avg_inference_time_ms)

In [ ]:
# Save final test metrics as a one-row CSV file
metrics_df = pd.DataFrame([{
    "model": config["model_name"],
    "iou": test_metrics["iou"],
    "f1": test_metrics["f1"],
    "precision": test_metrics["precision"],
    "recall": test_metrics["recall"],
    "params_m": param_count_million,
    "inference_time_ms": avg_inference_time_ms
}])

metrics_df.to_csv(output_dir / "metrics.csv", index=False)
metrics_df

In [ ]:
# Select a fixed set of shared test IDs for later comparison
shared_test_ids = test_dataset.sample_ids[:20]

with open(output_dir / "shared_test_ids.txt", "w", encoding="utf-8") as f:
    for sample_id in shared_test_ids:
        f.write(sample_id.replace(".png", "") + "\n")

print("Saved shared test IDs:")
print([sid.replace(".png", "") for sid in shared_test_ids[:10]])

In [ ]:
# Build a quick lookup from sample ID to dataset index
test_id_to_index = {sample_id: idx for idx, sample_id in enumerate(test_dataset.sample_ids)}

print("Total test IDs in lookup:", len(test_id_to_index))

In [ ]:
# Save predictions and sample visualizations for shared test IDs
best_model.eval()

for file_name in shared_test_ids:
    idx = test_id_to_index[file_name]
    sample = test_dataset[idx]

    image = sample["image"].unsqueeze(0).to(device)
    mask = sample["mask"].squeeze(0).numpy()
    sample_id = sample["id"]

    with torch.no_grad():
        logits = best_model(image)
        pred = binarize_predictions(logits, threshold=config["threshold"])

    pred_mask = pred.squeeze().cpu().numpy().astype(np.uint8) * 255

    # Save raw predicted mask
    pred_img = Image.fromarray(pred_mask)
    pred_img.save(test_predictions_dir / f"{sample_id}_pred.png")

    # Prepare visualizations
    image_np = sample["image"].numpy()
    time1_img = image_np[:3].transpose(1, 2, 0)
    time2_img = image_np[3:].transpose(1, 2, 0)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(time1_img)
    axes[0].set_title("Time 1")
    axes[0].axis("off")

    axes[1].imshow(time2_img)
    axes[1].set_title("Time 2")
    axes[1].axis("off")

    axes[2].imshow(mask, cmap="gray")
    axes[2].set_title("Ground Truth")
    axes[2].axis("off")

    axes[3].imshow(pred_mask, cmap="gray")
    axes[3].set_title("Prediction")
    axes[3].axis("off")

    plt.tight_layout()
    plt.savefig(sample_visuals_dir / f"{sample_id}_visual.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

print("Saved test predictions and sample visuals successfully.")

In [ ]:
# Preview one saved visualization
saved_visuals = sorted(sample_visuals_dir.glob("*.png"))

print("Number of saved visualizations:", len(saved_visuals))

if len(saved_visuals) > 0:
    preview_img = Image.open(saved_visuals[0])
    plt.figure(figsize=(12, 4))
    plt.imshow(preview_img)
    plt.axis("off")
    plt.title(saved_visuals[0].name)
    plt.show()

## Summary

This notebook completed the full U-Net baseline workflow:
- Loaded the predefined dataset split
- Built 6-channel inputs from bi-temporal imagery
- Trained a U-Net model with BCE + Dice loss
- Evaluated the model on the test set
- Saved standardized outputs for later model comparison

The exported files under `outputs/unet/` can now be reused by:
- `04_comparison.ipynb`
- `05_error_analysis.ipynb`
- `06_prediction.ipynb`

In [ ]:
# Display all key output files
print("Output directory:", output_dir)
print()

for path in sorted(output_dir.rglob("*")):
    print(path.relative_to(project_root))